# Mimi-VC Demo: Transformer-Based Voice Conversion

This notebook demonstrates the **learned Transformer** approach from our paper.
Unlike the retrieval method (FAISS+Smooth), the Transformer **learns** the target
speaker's acoustic patterns and can generalize to unseen phonetic contexts.

### How it works
1. **Record target speaker audio** (the voice to clone) — more data = better
2. **Train** a lightweight Transformer (~5M params) on the target speaker's tokens
3. **Record source audio** — the Transformer predicts target-style acoustic tokens
4. **Compare** both methods side-by-side (Retrieval vs Transformer)

### Requirements
- GPU runtime: **Runtime > Change runtime type > T4 GPU**
- Target speaker audio: **1+ minutes recommended** (more = better Transformer)

> Based on: *Mimi-VC: Voice Conversion via Disentangled Neural Audio Codec Token Manipulation*
> — Q.V. Nguyen, H. Nguyen, L.D. Vu

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers torch torchaudio faiss-cpu librosa soundfile matplotlib
print('Done.')

In [ ]:
# ============================================================
# 2. IMPORTS & LOAD MIMI
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os, time, random, base64
from collections import defaultdict
from pathlib import Path
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display, HTML
from google.colab import files, output
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading Mimi codec...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate
N_CB = mimi.config.num_quantizers
CB_SIZE = mimi.config.codebook_size
CB_DIM = mimi.config.codebook_dim
FRAME_RATE = mimi.config.frame_rate

print(f'Mimi: SR={SR}, {N_CB} codebooks x {CB_SIZE} (dim={CB_DIM}), {FRAME_RATE} Hz')

In [ ]:
# ============================================================
# 3. CORE FUNCTIONS
# ============================================================

def load_audio(path, target_sr=SR):
    wav, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != target_sr:
        wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr)
    return wav.astype(np.float32)


def encode(wav_np):
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()


def decode(codes):
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()


def get_codebook_embeddings(cb_idx=0):
    q = mimi.quantizer
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer: {type(q).__name__}')
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3:
                w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE:
                return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')


cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embeddings: {cb0_emb.shape}')

In [ ]:
# ============================================================
# 4. AUDIO RECORDER
# ============================================================

def record_audio(save_path, max_seconds=180, label="Audio"):
    """
    Record audio in the browser. Shows Start/Stop buttons with timer.
    Returns numpy audio at SR sample rate.
    """
    rec_id = f"rec_{random.randint(10000,99999)}"

    display(HTML(f'''
<div style="border:1px solid #ccc; padding:15px; border-radius:8px; max-width:500px; margin:10px 0;">
  <h4 style="margin-top:0;">Record {label}</h4>
  <button id="{rec_id}_start" onclick="{rec_id}_go()"
    style="background:#4CAF50; color:white; padding:10px 24px; border:none; border-radius:4px; cursor:pointer; font-size:14px; margin-right:8px;">
    Start Recording
  </button>
  <button id="{rec_id}_stop" onclick="{rec_id}_end()" disabled
    style="background:#f44336; color:white; padding:10px 24px; border:none; border-radius:4px; cursor:pointer; font-size:14px;">
    Stop Recording
  </button>
  <p id="{rec_id}_st" style="margin:10px 0 5px; font-weight:bold;">Click Start to begin.</p>
  <p id="{rec_id}_tm" style="font-family:monospace; font-size:18px; margin:0;">0:00</p>
  <audio id="{rec_id}_au" controls style="display:none; margin-top:10px; width:100%;"></audio>
</div>
<script>
var {rec_id}_mr,{rec_id}_ch,{rec_id}_iv,{rec_id}_s;
async function {rec_id}_go(){{var st=await navigator.mediaDevices.getUserMedia({{audio:{{sampleRate:48000,channelCount:1,echoCancellation:true,noiseSuppression:true}}}});{rec_id}_ch=[];{rec_id}_mr=new MediaRecorder(st,{{mimeType:'audio/webm;codecs=opus'}});{rec_id}_mr.ondataavailable=e=>{rec_id}_ch.push(e.data);{rec_id}_mr.onstop=()=>{{var b=new Blob({rec_id}_ch,{{type:'audio/webm'}});document.getElementById('{rec_id}_au').src=URL.createObjectURL(b);document.getElementById('{rec_id}_au').style.display='block';document.getElementById('{rec_id}_st').textContent='Done! Processing...';var r=new FileReader();r.onload=()=>google.colab.kernel.invokeFunction('nb.{rec_id}',[ r.result.split(',')[1] ],{{}});r.readAsDataURL(b);st.getTracks().forEach(t=>t.stop())}};{rec_id}_s=0;{rec_id}_iv=setInterval(()=>{{++{rec_id}_s;document.getElementById('{rec_id}_tm').textContent=Math.floor({rec_id}_s/60)+':'+String({rec_id}_s%60).padStart(2,'0');if({rec_id}_s>={max_seconds}){rec_id}_end()}},1000);{rec_id}_mr.start(1000);document.getElementById('{rec_id}_start').disabled=true;document.getElementById('{rec_id}_stop').disabled=false;document.getElementById('{rec_id}_st').textContent='Recording... speak now!';document.getElementById('{rec_id}_st').style.color='#f44336'}}
function {rec_id}_end(){{clearInterval({rec_id}_iv);if({rec_id}_mr&&{rec_id}_mr.state==='recording'){rec_id}_mr.stop();document.getElementById('{rec_id}_start').disabled=false;document.getElementById('{rec_id}_stop').disabled=true}}
</script>
'''))

    result = {'done': False}
    webm_path = save_path.replace('.wav', '.webm')

    def on_save(b64):
        with open(webm_path, 'wb') as f:
            f.write(base64.b64decode(b64))
        result['done'] = True

    output.register_callback(f'nb.{rec_id}', on_save)
    print(f'Waiting for {label} recording... (click Start, speak, click Stop)')
    while not result['done']:
        time.sleep(0.5)

    wav = load_audio(webm_path)
    sf.write(save_path, wav, SR)
    print(f'Recorded {len(wav)/SR:.1f}s')
    return wav

print('Recorder ready.')

In [ ]:
# ============================================================
# 5. ACOUSTIC TRANSLATOR (TRANSFORMER MODEL)
# ============================================================

class AcousticTranslator(nn.Module):
    """
    Predicts acoustic tokens (CB1-7) from semantic tokens (CB0)
    for a target speaker.

    Architecture: semantic embedding -> causal Transformer -> 7 output heads
    """
    def __init__(self, n_sem=2048, n_acou=2048, n_cb=7,
                 d_model=256, n_heads=4, n_layers=4, max_len=1024, dropout=0.1):
        super().__init__()
        self.n_cb = n_cb
        self.sem_embed = nn.Embedding(n_sem, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.heads = nn.ModuleList(
            [nn.Linear(d_model, n_acou) for _ in range(n_cb)]
        )

    def forward(self, sem_tokens):
        B, T = sem_tokens.shape
        x = self.sem_embed(sem_tokens) + self.pos_embed[:, :T]
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return [h(x) for h in self.heads]


print(f'AcousticTranslator defined.')
print(f'  Input: semantic tokens (CB0) at 12.5 Hz')
print(f'  Output: 7 acoustic token sequences (CB1-7)')
print(f'  Architecture: 4-layer causal Transformer, d=256, 4 heads')

---
## Step 1: Record Target Speaker

Record the **target speaker** — the voice you want to clone.

**For the Transformer method, more data matters:**
- **30s**: basic voice capture (Transformer will underperform retrieval)
- **1-2 min**: good quality, Transformer starts to generalize
- **3+ min**: best results, Transformer can outperform retrieval

Speak naturally with varied sentences for diverse phonetic coverage.

In [ ]:
# ============================================================
# 6. RECORD TARGET SPEAKER
# ============================================================

target_wav = record_audio(
    save_path='/tmp/target_speaker.wav',
    max_seconds=300,  # up to 5 minutes
    label='Target Speaker (the voice to clone)'
)

target_wavs = [target_wav]
target_dur = len(target_wav) / SR

print(f'\nTarget speaker audio: {target_dur:.1f}s')
if target_dur < 30:
    print('WARNING: Very short. Transformer needs 1+ min for good results.')
elif target_dur < 60:
    print('TIP: 1+ minutes would improve Transformer quality.')
else:
    print('Good amount of data for Transformer training.')

print('\nPlayback:')
display(PlayAudio(target_wav[:int(SR*15)], rate=SR))

In [ ]:
# ============================================================
# 6b. (ALTERNATIVE) Upload target audio instead
# ============================================================
# Skip this cell if you recorded above.

print('Upload target speaker audio file(s):')
uploaded = files.upload()

target_wavs = []
for fname, data in uploaded.items():
    tmp = f'/tmp/target_{fname}'
    with open(tmp, 'wb') as f:
        f.write(data)
    target_wavs.append(load_audio(tmp))
    print(f'  {fname}: {len(target_wavs[-1])/SR:.1f}s')

target_dur = sum(len(w) for w in target_wavs) / SR
print(f'Total: {target_dur:.1f}s')
display(PlayAudio(target_wavs[0][:int(SR*15)], rate=SR))

---
## Step 2: Encode & Train

1. Encode target audio into Mimi tokens
2. Build FAISS retrieval index (for comparison)
3. Train Transformer on target speaker's token patterns

In [ ]:
# ============================================================
# 7. ENCODE TARGET AUDIO
# ============================================================

CHUNK_SEC = 30
chunk_samples = int(CHUNK_SEC * SR)

# Split into chunks
all_chunks = []
for wav in target_wavs:
    for i in range(0, len(wav), chunk_samples):
        chunk = wav[i:i + chunk_samples]
        if len(chunk) > SR:
            all_chunks.append(chunk)

# Encode all chunks
print(f'Encoding {len(all_chunks)} chunks...')
target_codes_list = []
for i, chunk in enumerate(all_chunks):
    codes = encode(chunk)
    target_codes_list.append(codes)
    print(f'  Chunk {i+1}/{len(all_chunks)}: {codes.shape[2]} frames')

# Concatenate all codes
all_target_codes = torch.cat(target_codes_list, dim=2)  # [1, 8, T_total]
T_total = all_target_codes.shape[2]
print(f'\nTotal: {T_total} frames ({T_total/FRAME_RATE:.1f}s)')

In [ ]:
# ============================================================
# 8. BUILD FAISS INDEX (for comparison)
# ============================================================

all_sem = []
all_full = []
for t in range(T_total):
    all_sem.append(all_target_codes[0, 0, t].item())
    all_full.append(all_target_codes[0, :, t].numpy())

sem_arr = np.array(all_sem)
full_arr = np.stack(all_full)

# FAISS index
tgt_emb = cb0_emb[sem_arr].astype(np.float32)
faiss.normalize_L2(tgt_emb)
faiss_index = faiss.IndexFlatIP(cb0_emb.shape[1])
faiss_index.add(tgt_emb)

print(f'FAISS index: {faiss_index.ntotal} vectors')


def retrieval_vc(src_codes, k=8, alpha=0.5):
    """FAISS+Smooth retrieval VC for comparison."""
    T = src_codes.shape[2]
    src_sem = src_codes[0, 0].numpy()
    src_emb = cb0_emb[src_sem].astype(np.float32)
    faiss.normalize_L2(src_emb)
    dists, indices = faiss_index.search(src_emb, k)

    selected = np.zeros(T, dtype=np.int64)
    selected[0] = indices[0, 0]
    for t in range(1, T):
        cands = indices[t]
        sem_s = dists[t]
        tmp_s = 1.0 / (1.0 + np.abs(cands - selected[t-1]).astype(np.float32))
        selected[t] = cands[np.argmax(alpha * sem_s + (1-alpha) * tmp_s)]

    out = src_codes.clone()
    for t in range(T):
        out[0, :, t] = torch.tensor(full_arr[selected[t]])
    return out


print('Retrieval method ready.')

In [ ]:
# ============================================================
# 9. TRAIN ACOUSTIC TRANSLATOR
# ============================================================

# --- Hyperparameters ---
# Scale training steps with data amount
base_steps = 500
data_factor = max(1.0, T_total / 500)  # more data -> more steps
N_STEPS = min(int(base_steps * data_factor), 3000)
BATCH_SIZE = 8
SEQ_LEN = 64
LR = 3e-4

print(f'Training config:')
print(f'  Data: {T_total} frames ({T_total/FRAME_RATE:.1f}s)')
print(f'  Steps: {N_STEPS}, Batch: {BATCH_SIZE}, Seq len: {SEQ_LEN}')
print(f'  LR: {LR}, Optimizer: AdamW')

# Prepare training data
train_sem = all_target_codes[0, 0]     # [T_total]
train_acou = all_target_codes[0, 1:]   # [7, T_total]

# Initialize model
translator = AcousticTranslator(
    n_sem=CB_SIZE, n_acou=CB_SIZE, n_cb=N_CB-1
).to(device)
n_params = sum(p.numel() for p in translator.parameters())
print(f'  Model: {n_params/1e6:.1f}M parameters\n')

optimizer = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_STEPS)

# --- Training loop ---
losses = []
translator.train()
t0 = time.time()

for step in range(N_STEPS):
    # Random batches
    starts = torch.randint(0, max(1, T_total - SEQ_LEN), (BATCH_SIZE,))
    sem_b = torch.stack([train_sem[s:s+SEQ_LEN] for s in starts]).to(device)
    acou_b = torch.stack([train_acou[:, s:s+SEQ_LEN] for s in starts]).to(device)

    logits = translator(sem_b)
    loss = sum(
        F.cross_entropy(lg.reshape(-1, CB_SIZE), acou_b[:, i].reshape(-1))
        for i, lg in enumerate(logits)
    ) / len(logits)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(translator.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())

    if (step + 1) % max(1, N_STEPS // 10) == 0:
        elapsed = time.time() - t0
        print(f'  Step {step+1:4d}/{N_STEPS} | Loss: {loss.item():.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}s')

train_time = time.time() - t0
translator.eval()

# Plot loss
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(losses, color='#7030A0', linewidth=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title(f'Transformer Training — {N_STEPS} steps in {train_time:.1f}s')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nTraining complete: {train_time:.1f}s, final loss: {losses[-1]:.4f}')

---
## Step 3: Record Source & Convert

Record or speak something — it will be converted using **both methods** for comparison:
1. **FAISS+Smooth** (retrieval, no training)
2. **Transformer** (learned, just trained above)

In [ ]:
# ============================================================
# 10. RECORD SOURCE & CONVERT WITH BOTH METHODS
# ============================================================

source_wav = record_audio(
    save_path='/tmp/source_audio.wav',
    max_seconds=120,
    label='Source Audio (the speech to convert)'
)

src_dur = len(source_wav) / SR
print(f'\nSource: {src_dur:.1f}s')

# Encode source
src_codes = encode(source_wav)
print(f'Encoded: {src_codes.shape[2]} frames')

# --- Method 1: Retrieval (FAISS+Smooth) ---
print('\nConverting with FAISS+Smooth...')
t0 = time.time()
ret_codes = retrieval_vc(src_codes)
ret_wav = decode(ret_codes)
ret_time = time.time() - t0
print(f'  Done in {ret_time:.2f}s (RTF={ret_time/src_dur:.4f})')

# --- Method 2: Transformer ---
@torch.no_grad()
def transformer_convert(src_codes, temperature=0.8, top_k=50):
    sem = src_codes[0, 0:1].to(device)
    logits_list = translator(sem)
    out = src_codes.clone()
    for i, logits in enumerate(logits_list):
        logits = logits / temperature
        if top_k > 0:
            topk_v, _ = torch.topk(logits, top_k, dim=-1)
            logits[logits < topk_v[:, :, -1:]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        pred = torch.multinomial(probs.reshape(-1, CB_SIZE), 1).reshape(1, -1)
        out[0, i + 1] = pred.cpu()
    return out

print('\nConverting with Transformer...')
t0 = time.time()
torch.manual_seed(42)
trans_codes = transformer_convert(src_codes, temperature=0.8)
trans_wav = decode(trans_codes)
trans_time = time.time() - t0
print(f'  Done in {trans_time:.2f}s (RTF={trans_time/src_dur:.4f})')

# --- Playback ---
print('\n' + '='*60)
print('RESULTS')
print('='*60)

print('\n1. Original source:')
display(PlayAudio(source_wav, rate=SR))

print('\n2. FAISS+Smooth (retrieval, no training):')
display(PlayAudio(ret_wav, rate=SR))

print('\n3. Transformer (learned, trained on target speaker):')
display(PlayAudio(trans_wav, rate=SR))

print('\n4. Target speaker reference:')
display(PlayAudio(target_wavs[0][:int(SR*15)], rate=SR))

# Save
sf.write('/content/converted_retrieval.wav', ret_wav, SR)
sf.write('/content/converted_transformer.wav', trans_wav, SR)
print('\nSaved: converted_retrieval.wav, converted_transformer.wav')

In [ ]:
# ============================================================
# 10b. (ALTERNATIVE) Upload source audio instead
# ============================================================
# Skip if you recorded above.

print('Upload source audio:')
up = files.upload()
fname = list(up.keys())[0]
tmp = f'/tmp/source_{fname}'
with open(tmp, 'wb') as f:
    f.write(up[fname])
source_wav = load_audio(tmp)
src_dur = len(source_wav) / SR
print(f'Loaded: {fname} ({src_dur:.1f}s)')

# Encode
src_codes = encode(source_wav)

# Retrieval
print('Converting with FAISS+Smooth...')
ret_codes = retrieval_vc(src_codes)
ret_wav = decode(ret_codes)

# Transformer
print('Converting with Transformer...')
torch.manual_seed(42)
trans_codes = transformer_convert(src_codes, temperature=0.8)
trans_wav = decode(trans_codes)

print('\n1. Original:')
display(PlayAudio(source_wav, rate=SR))
print('\n2. FAISS+Smooth:')
display(PlayAudio(ret_wav, rate=SR))
print('\n3. Transformer:')
display(PlayAudio(trans_wav, rate=SR))
print('\n4. Target reference:')
display(PlayAudio(target_wavs[0][:int(SR*15)], rate=SR))

sf.write('/content/converted_retrieval.wav', ret_wav, SR)
sf.write('/content/converted_transformer.wav', trans_wav, SR)

---
## Step 4: Visualize & Compare

In [ ]:
# ============================================================
# 11. WAVEFORM COMPARISON
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(14, 10), constrained_layout=True)

wavs = [source_wav, ret_wav, trans_wav, target_wavs[0][:int(SR*30)]]
titles = ['Source (original)', 'FAISS+Smooth (retrieval)',
          'Transformer (learned)', 'Target Speaker (reference)']
colors = ['#4472C4', '#548235', '#7030A0', '#C55A11']

for ax, wav, title, color in zip(axes, wavs, titles, colors):
    t_ax = np.arange(len(wav)) / SR
    ax.plot(t_ax, wav, color=color, linewidth=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, max(t_ax[-1], 0.1))
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Mimi-VC: Retrieval vs Transformer', fontsize=13)
plt.savefig('/content/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 12. TOKEN HEATMAP COMPARISON
# ============================================================

vis_len = min(int(SR * 10), len(source_wav), len(ret_wav), len(trans_wav))
tgt_vis = encode(target_wavs[0][:vis_len])
L = min(src_codes.shape[2], ret_codes.shape[2], trans_codes.shape[2], tgt_vis.shape[2], 200)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), constrained_layout=True)

code_list = [
    ('Source (original)', src_codes[0, :, :L]),
    ('FAISS+Smooth (retrieval)', ret_codes[0, :, :L]),
    ('Transformer (learned)', trans_codes[0, :, :L]),
    ('Target Speaker (reference)', tgt_vis[0, :, :L]),
]

for ax, (title, codes) in zip(axes, code_list):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('CB')
    ax.set_title(title, fontsize=11)
    ax.set_yticks(range(N_CB))
    ax.set_yticklabels([f'CB{i}{" (sem)" if i==0 else ""}' for i in range(N_CB)])
    plt.colorbar(im, ax=ax, shrink=0.6)

axes[-1].set_xlabel('Frame (12.5 Hz)')
plt.suptitle('Token Heatmap: Retrieval vs Transformer', fontsize=13)
plt.savefig('/content/tokens_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# CB0 preservation check
src_cb0 = src_codes[0, 0, :L].numpy()
ret_cb0 = ret_codes[0, 0, :L].numpy()
trans_cb0 = trans_codes[0, 0, :L].numpy()

# Retrieval replaces all CBs (including CB0), Transformer keeps source CB0
ret_cb0_match = np.mean(src_cb0 == ret_cb0) * 100
trans_cb0_match = np.mean(src_cb0 == trans_cb0) * 100
print(f'CB0 preservation (content tokens kept from source):')
print(f'  Retrieval:   {ret_cb0_match:.1f}% (replaces all CBs including CB0)')
print(f'  Transformer: {trans_cb0_match:.1f}% (keeps source CB0, predicts CB1-7)')

In [ ]:
# ============================================================
# 13. TRANSFORMER TEMPERATURE SWEEP
# ============================================================
# Try different temperatures to hear the effect on output.

temps = [0.2, 0.5, 0.8, 1.0, 1.5]

print('Sampling temperature comparison:')
print('  Lower = more deterministic, Higher = more diverse\n')

for temp in temps:
    torch.manual_seed(42)
    tc = transformer_convert(src_codes, temperature=temp)
    tw = decode(tc)
    print(f'Temperature = {temp}:')
    display(PlayAudio(tw, rate=SR))
    print()

In [ ]:
# ============================================================
# 14. DOWNLOAD RESULTS
# ============================================================

print('Final summary:')
print(f'  Target speaker: {target_dur:.1f}s of training data')
print(f'  Transformer: {n_params/1e6:.1f}M params, trained in {train_time:.1f}s')
print(f'  Source audio: {src_dur:.1f}s')

print('\nDownloading results...')
files.download('/content/converted_retrieval.wav')
files.download('/content/converted_transformer.wav')
print('Done!')

---
## (Optional) Convert More

Both models are still loaded. Record more audio to convert.

In [ ]:
# ============================================================
# 15. CONVERT MORE RECORDINGS
# ============================================================

extra_wav = record_audio(
    save_path=f'/tmp/extra_{random.randint(1,9999)}.wav',
    max_seconds=120,
    label='More Source Audio'
)

extra_codes = encode(extra_wav)

# Both methods
ret_out = decode(retrieval_vc(extra_codes))
torch.manual_seed(42)
trans_out = decode(transformer_convert(extra_codes, temperature=0.8))

print('\n1. Original:')
display(PlayAudio(extra_wav, rate=SR))
print('\n2. FAISS+Smooth:')
display(PlayAudio(ret_out, rate=SR))
print('\n3. Transformer:')
display(PlayAudio(trans_out, rate=SR))

eid = random.randint(1,9999)
sf.write(f'/content/extra_retrieval_{eid}.wav', ret_out, SR)
sf.write(f'/content/extra_transformer_{eid}.wav', trans_out, SR)
files.download(f'/content/extra_retrieval_{eid}.wav')
files.download(f'/content/extra_transformer_{eid}.wav')